In [ ]:
!rm -rf /root/.cache/kaggle
!pip install trl deepspeed mlflow bitsandbytes>=0.46.1 --quiet
!pip install -U nvtx

## It is important to select "pin to the original environment" in Settings !!!

In [ ]:
%%writefile accel_test.py

import os
from datasets import load_dataset
from datetime import datetime


os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

def get_model(model_name, AutoModelForCausalLM, BitsAndBytesConfig, dtype):
    quantization_config = BitsAndBytesConfig(
      # Load the model with 4-bit quantization
      load_in_4bit=True,
      # Use double quantization
      bnb_4bit_use_double_quant=True,
      # Use 4-bit Normal Float for storing the base model weights in GPU memory
      bnb_4bit_quant_type="nf4",
      # De-quantize the weights to 16-bit (Brain) float before the forward/backward pass
      bnb_4bit_compute_dtype=dtype,
      # Must be the same as dtype of the models
      bnb_4bit_quant_storage=dtype
    )


    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=dtype,
        trust_remote_code=True,
        quantization_config=quantization_config,
        force_download=True
        #attn_implementation="flash_attention_2"
    )

    return model


def get_tokenizers(model_name, instruct_model_name, AutoTokenizer):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

    tokenizer.pad_token = tokenizer.eos_token  # Set padding token
    tokenizer.padding_side = "right"  # Padding on the right for generation

    return tokenizer, instruct_tokenizer


def get_datasets(
    dataset_name,
    subdataset_name,
    number_samples,
    test_size=0.1,
    shuffle_seed=42,
    streaming=True,
):
    
    #mem_before = Process(getpid()).memory_info().rss / (1024 * 1024)

    train_size = 1 - test_size
    train_samples = int(train_size * number_samples)

    if streaming:
        #logger.info(f"Loading dataset{dataset_name} and shuffle...")
    
        dataset = load_dataset(
            dataset_name,
            'SFT',
            split=subdataset_name,
            streaming=streaming
            ).take(number_samples).shuffle(seed=42, buffer_size=number_samples)

        #logger.info(f"Splitting into train and dev dataset...")

        train_dataset = dataset.take(train_samples)
        dev_dataset = dataset.skip(train_samples)
    else:
        
        #logger.info(f"Loading dataset{dataset_name} and shuffle...")
        
        dataset = load_dataset(
            dataset_name,
            'SFT',
            split=subdataset_name,
            streaming=streaming
            ).select(range(number_samples)).shuffle(seed=42, buffer_size=number_samples)

        train_dataset = dataset.select(range(train_samples))
        dev_dataset = dataset.select(range(train_samples, number_samples))

    
    #mem_after = Process(getpid()).memory_info().rss / (1024 * 1024)

    #logger.info(f"RAM memory used: {(mem_after - mem_before)} MB") 
    
    
    return train_dataset, dev_dataset


def format_chat_template(example: dict) -> dict:
    """Format the messages using the chat template"""
    if "messages" in example.keys():
        # SmolTalk2 format
        messages = example["messages"]
    else:
        # Custom format - adapt as needed
        messages = [
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["response"]}
        ]
    
    # Apply chat template
    text = instruct_tokenizer.apply_chat_template(
        messages, 
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}


def format_dataset(dataset, format_callable, column_to_maintain, instruct_tokenizer):
    dataset = dataset.map(format_callable, fn_kwargs = {"instruct_tokenizer": instruct_tokenizer}, batched=True)

    cols = [col for col in next(iter(dataset)).keys() if col != column_to_maintain]

    dataset = dataset.remove_columns(cols)

    return dataset


def preprocess(samples, instruct_tokenizer):
    batch = []
    for conversation in samples["messages"]:
        batch.append(instruct_tokenizer.apply_chat_template(conversation, tokenize=False))
    return {"content": batch}


def write_loss(trainer, loss_file="loss.csv"):
    # Only main process writes result
    history = trainer.state.log_history

    eval_loss = history[-2]['eval_loss']
    train_loss = history[-1]['train_loss']

    with open(loss_file, "w") as fh:
        fh.write(f"{eval_loss}, {train_loss}")

def train(
    model_name,
    instruct_model_name,
    new_model_name,
    dataset_name,
    subdataset_name,
    number_samples,
    learning_rate,
    num_train_epochs,
    resume_from_checkpoint,
    max_steps,
    per_device_train_batch_size,
    gradient_accumulation_steps
):
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from trl import SFTConfig, SFTTrainer
    from peft import LoraConfig
    from accelerate import PartialState
    import torch

    state = PartialState()

    state.print("Loading training and development set...")

    train_dataset, dev_dataset = get_datasets(
        dataset_name,
        subdataset_name,
        number_samples,
        test_size=0.1,
        shuffle_seed=42
    )

    state.print("Loading tokenizers...")
    tokenizer, instruct_tokenizer = get_tokenizers(model_name, instruct_model_name, AutoTokenizer)
    
    #Apply formatting
    state.print("Formatting datasets...")
    formatted_train_dataset = format_dataset(train_dataset, preprocess, "content", instruct_tokenizer)
    formatted_dev_dataset =  format_dataset(dev_dataset, preprocess, "content", instruct_tokenizer)

    state.print(f"Loading model {model_name}...")
    model = get_model(model_name, AutoModelForCausalLM, BitsAndBytesConfig, torch.bfloat16)

    # Setting the mlflow experiment. Note only rank 0 set it
    if torch.distributed.get_rank() == 0:
        import mlflow
        mlflow.set_experiment(f"{new_model_name}_experiment")

    now = datetime.now().strftime('%Y-%m-%d-%H-%M-%s')

    # Configure training parameters
    state.print(f"Configure training parameters...")
    training_config = SFTConfig(
        # Model and data
        dataset_text_field="content",
        max_length=512,#2048
        
        # Training hyperparameters
        per_device_train_batch_size=per_device_train_batch_size,  # Adjust based on your GPU memory
        gradient_accumulation_steps=gradient_accumulation_steps,
        learning_rate=learning_rate,
        num_train_epochs=num_train_epochs,
        max_steps=max_steps,
        
        # Optimization
        warmup_steps=50,
        weight_decay=0.01,
        optim="adamw_torch",
        
        # Logging and saving for checkpoints
        logging_steps=2,
        save_strategy='steps',
        save_steps=2,
        eval_strategy="steps",
        eval_steps=2,
        save_total_limit=2,
        
        # Memory optimization
        #dataloader_num_workers=0,
        #group_by_length=True,  # Group similar length sequences
        
        # Hugging Face Hub integration
        #push_to_hub=False,  # Set to True to upload to Hub
        #hub_model_id=f"your-username/{new_model_name}",
        
        # Experiment tracking
        report_to=["mlflow"],
        run_name=f"{new_model_name}_{now}",
        output_dir=f"{new_model_name}"
    )

    number_of_devices = torch.distributed.get_world_size()
    state.print(f"Number of GPUs: {number_of_devices}")
    state.print(f"Global Batch Size: {per_device_train_batch_size * number_of_devices * gradient_accumulation_steps}")

    state.print("Configure LoRa parameters...")
    peft_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules = 'all-linear'
    )

    state.print("Configure LoRa trainer...")
    lora_trainer = SFTTrainer(
        model=model,
        train_dataset=formatted_train_dataset,
        eval_dataset=formatted_dev_dataset,# dataset with a "text" field or messages + dataset_text_field in config
        args=training_config,
        peft_config=peft_config  # << enable LoRA
    )

    if resume_from_checkpoint == 1:
        state.print("Resume fine tuning from last checkpoint")
        resume_from_checkpoint = True        
    else:
        resume_from_checkpoint = False
        
    
    lora_trainer.train(resume_from_checkpoint=resume_from_checkpoint)

    # Writing the loss in a file. Note that only rank 0 will write
    if torch.distributed.get_rank() == 0:
        write_loss(lora_trainer)


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()


    parser.add_argument("--model_name", help="Base Model") 
    parser.add_argument("--instruct_model_name", help = "Instructed Model") 
    parser.add_argument("--new_model_name", help = "Fine Tuned Model") 
    parser.add_argument("--dataset_name", help = "Dataset Name")
    parser.add_argument("--subdataset_name", help = "Sub-Dataset Name")
    parser.add_argument("--number_samples", help = "Number of Samples") 
    parser.add_argument("--learning_rate", help="Learning Rate")
    parser.add_argument("--epochs", help="Number of epochs. If max_steps specified then epochs is not used.")
    parser.add_argument("--max_steps", help="Number of optimizers updates")
    parser.add_argument("--resume_from_checkpoint", help="1 to resume finetuning from last checkpoint")
    parser.add_argument("--per_device_train_batch_size", help="Batch size per GPU")
    parser.add_argument("--gradient_accumulation_steps", help="Number of update steps to accumulate gradients before performing a weight update pass. Useful to simulate larger batch sizes witout additional memory")

    
    args = parser.parse_args()


    args = (
        args.model_name,
        args.instruct_model_name,
        args.new_model_name,
        args.dataset_name,
        args.subdataset_name,
        int(args.number_samples),
        float(args.learning_rate),
        int(args.epochs),
        int(args.resume_from_checkpoint),
        int(args.max_steps),
        int(args.per_device_train_batch_size),
        int(args.gradient_accumulation_steps)
    )

    
    train(*args)

In [ ]:
%%writefile deepspeed_config.yaml

compute_environment: LOCAL_MACHINE                                                                                                                                           
debug: false
deepspeed_config:
  deepspeed_multinode_launcher: standard
  gradient_accumulation_steps: 2
  offload_optimizer_device: none
  offload_param_device: none
  zero3_init_flag: true
  zero3_save_16bit_model: true
  zero_stage: 3
distributed_type: DEEPSPEED
downcast_bf16: 'no'
machine_rank: 0
main_training_function: main
mixed_precision: bf16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false


In [ ]:
import optuna
import subprocess


def read_loss(loss_file="loss.csv"):
    # Function to read the loss of the trial 

    with open(loss_file, "r") as fh:
        losses = fh.read()

    losses = losses.split(", ")
    return float(losses[0]), float(losses[1])


def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 5e-5, 1e-04, log=True)
    epochs = trial.suggest_int("epochs", 1, 1)


    cmd = [
        "accelerate",
        "launch",
        "--config_file",
        conf_yaml,
        source_file,
        "--model_name",
        model_name,
        "--instruct_model_name",
        instruct_model_name,
        "--new_model_name",
        new_model_name,
        "--dataset_name",
        dataset_name,
        "--subdataset_name",
        subdataset_name,
        "--number_samples",
        number_samples,
        "--learning_rate",
        str(learning_rate),
        "--epochs",
        str(epochs),
        "--resume_from_checkpoint",
        resume_from_checkpoint,
        "--max_steps",
        max_steps,
        "--per_device_train_batch_size",
        per_device_train_batch_size,
        "--gradient_accumulation_steps",
        gradient_accumulation_steps
        
    ]
    
    subprocess.run(cmd, check=True)

    eval_loss, train_loss = read_loss()

    return train_loss, eval_loss - train_loss


conf_yaml = "deepspeed_config.yaml"
source_file =  "accel_test.py"

# Parameters
model_name = "HuggingFaceTB/SmolLM3-3B-Base"
instruct_model_name = "HuggingFaceTB/SmolLM3-3B"
new_model_name = "SmolLM3-Custom-SFT"
dataset_name = "HuggingFaceTB/smoltalk2"
subdataset_name = "smoltalk_everyday_convs_reasoning_Qwen3_32B_think"
number_samples = "10"
resume_from_checkpoint = "0"
max_steps = "20"
per_device_train_batch_size = "2"
gradient_accumulation_steps = "2"

# optuna parameters

optuna_db = "optuna.db"
optuna_storage = f"sqlite:///{optuna_db}"

study = optuna.create_study(
    directions=["minimize", "minimize"],
    storage=optuna_storage,
    study_name=f"{new_model_name}_study",
    load_if_exists=True
)
study.optimize(objective, n_trials=2)

print(study.best_trials)

In [ ]:
#!tar -czf /kaggle/working/SmolLM3-Custom-SFT/checkpoint-31.tar.gz /kaggle/working/SmolLM3-Custom-SFT/checkpoint-31/

In [ ]:
#!ls /kaggle/working/SmolLM3-Custom-SFT/

## Post-Processing training results

### Optuna
* studies
* trials
* trial_params
* trial_values
* version_info
### MLFlow
* experiments
* runs
* metrics
* latest_metrics
* params
* tags
* model_versions
* experiment_tags
* registered_models

In [ ]:
import sqlite3
import pandas as pd

def show_results(db, table_name):
    # Connect to the SQLite database
    conn = sqlite3.connect(db)

    # SQL query
    query = f"select * from {table_name}"

    # Load query results into a pandas DataFrame
    df = pd.read_sql_query(query, conn)

    # Show results
    print(df)

    # Close connection
    conn.close()

In [ ]:
show_results("mlflow.db", "metrics")